# llm-tta TPU Launcher

这个 notebook 不再直接实现 `llm-tta` 的核心计算，而是统一调用脚本：

- `run_llm_tta_xla.py`

适合 TPU v4-8 场景。默认使用 `PyTorch/XLA`，并把结果输出到 `save/` 目录。

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

PROJECT_DIR = Path.cwd()
SCRIPT_PATH = PROJECT_DIR / "run_llm_tta_xla.py"
OUTPUT_DIR = PROJECT_DIR / "save"

print(f"PROJECT_DIR = {PROJECT_DIR}")
print(f"SCRIPT_PATH = {SCRIPT_PATH}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")
assert SCRIPT_PATH.exists(), f"Script not found: {SCRIPT_PATH}"

## 运行参数

对 TPU v4-8，下面默认给 `XLA_PROCESSES = 4`。

这通常对应一颗 chip 一个 worker 的常见用法。如果你的环境已经验证更适合 `8`，把它改成 `8` 即可。

In [ ]:
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DEVICE = "xla"
PRECISION = "bf16"
SAMPLES_PER_TYPE = 200
MAX_LEN = 512
SEED = 42
RUN_NAME = "llama3_tpu_v4_8"
XLA_PROCESSES = 4
SKIP_PLOTS = False

print("MODEL_NAME =", MODEL_NAME)
print("DEVICE =", DEVICE)
print("PRECISION =", PRECISION)
print("SAMPLES_PER_TYPE =", SAMPLES_PER_TYPE)
print("MAX_LEN =", MAX_LEN)
print("SEED =", SEED)
print("RUN_NAME =", RUN_NAME)
print("XLA_PROCESSES =", XLA_PROCESSES)
print("SKIP_PLOTS =", SKIP_PLOTS)

In [ ]:
env = os.environ.copy()
env["PJRT_DEVICE"] = "TPU"

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--device", DEVICE,
    "--use-xla-spawn",
    "--xla-processes", str(XLA_PROCESSES),
    "--model-name", MODEL_NAME,
    "--precision", PRECISION,
    "--samples-per-type", str(SAMPLES_PER_TYPE),
    "--max-len", str(MAX_LEN),
    "--seed", str(SEED),
    "--run-name", RUN_NAME,
    "--output-dir", str(OUTPUT_DIR),
]

if SKIP_PLOTS:
    cmd.append("--skip-plots")

print("Command:")
print(" ".join(shlex.quote(part) for part in cmd))

## 执行

运行下面这个单元格后，会直接调用脚本并把日志输出到 notebook。

In [ ]:
result = subprocess.run(cmd, cwd=str(PROJECT_DIR), env=env, check=False)
print("Return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("llm-tta TPU run failed, please check the logs above.")

## 查看结果

默认结果位置：

- `save/<RUN_NAME>.csv`
- `save/<RUN_NAME>_gradient_dot_product.png`
- `save/<RUN_NAME>_by_type.png`

In [ ]:
csv_path = OUTPUT_DIR / f"{RUN_NAME}.csv"
dot_plot_path = OUTPUT_DIR / f"{RUN_NAME}_gradient_dot_product.png"
type_plot_path = OUTPUT_DIR / f"{RUN_NAME}_by_type.png"

print("CSV exists:", csv_path.exists(), csv_path)
print("Dot plot exists:", dot_plot_path.exists(), dot_plot_path)
print("Type plot exists:", type_plot_path.exists(), type_plot_path)